# 01 — Prepare training and test gene sets

This notebook prepares the positive and negative gene sets used for supervised
learning. It:

1. loads the normalized *C. elegans* expression matrix and OXPHOS gene annotation;
2. defines the positive gene set and complex-specific positive subsets;
3. creates one balanced, held-out test set;
4. generates ten balanced negative training sets;
5. constructs the informed bagging collections; and
6. saves the resulting Python objects as pickle files.

All random operations use a fixed seed to make the procedure reproducible.


## 1. Imports and configuration

Paths are defined relative to the repository root. Update `EXPRESSION_FILE` and
`GENE_ANNOTATION_FILE` only when the input files are stored elsewhere.


In [ ]:
from pathlib import Path
import pickle
import random

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


# Reproducibility
RANDOM_SEED = 123
TEST_SIZE = 0.25
N_NEGATIVE_SETS = 10

# Repository paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "results" / "prepared_lists"

EXPRESSION_FILE = DATA_DIR / "boeck_tmm.csv"
GENE_ANNOTATION_FILE = DATA_DIR / "strong_positives_first_iteration.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Helper functions


In [ ]:
def validate_unique_index(expression_df):
    """Ensure that the expression matrix contains one row per gene."""
    if expression_df.index.has_duplicates:
        duplicated = expression_df.index[expression_df.index.duplicated()].unique()
        raise ValueError(
            f"The expression matrix contains {len(duplicated)} duplicated gene IDs."
        )


def informed_bagging(
    expression_df,
    positive_genes,
    negative_pool,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
):
    """Create a balanced held-out test set.

    Positive genes are split into training and test partitions. The test set is
    completed with the same number of genes sampled from the negative pool.

    Returns
    -------
    X_test : numpy.ndarray
        Expression features for held-out genes.
    y_test : numpy.ndarray
        Binary labels, where 1 indicates a positive OXPHOS gene.
    test_genes : list[str]
        Gene IDs in the order used in `X_test` and `y_test`.
    positive_train : list[str]
        Positive genes retained for training.
    positive_test : list[str]
        Positive genes assigned to the test set.
    negative_test : set[str]
        Negative genes assigned to the test set.
    """
    positive_genes = list(positive_genes)
    positive_set = set(positive_genes)

    positive_train, positive_test = train_test_split(
        positive_genes,
        test_size=test_size,
        random_state=random_state,
        shuffle=True,
    )

    eligible_negatives = [
        gene for gene in negative_pool if gene not in positive_set
    ]

    if len(eligible_negatives) < len(positive_test):
        raise ValueError(
            "The negative pool is too small to construct a balanced test set."
        )

    rng = random.Random(random_state)
    negative_test = rng.sample(eligible_negatives, len(positive_test))

    test_genes = positive_test + negative_test
    y_test = np.array(
        [1] * len(positive_test) + [0] * len(negative_test),
        dtype=int,
    )

    permutation = np.random.RandomState(random_state).permutation(len(test_genes))
    test_genes = [test_genes[index] for index in permutation]
    y_test = y_test[permutation]
    X_test = expression_df.loc[test_genes].to_numpy()

    return (
        X_test,
        y_test,
        test_genes,
        positive_train,
        positive_test,
        set(negative_test),
    )


def create_training_dictionaries(
    expression_df,
    positive_train,
    positive_subsets,
    negative_training_sets,
    random_state=RANDOM_SEED,
):
    """Build balanced training collections.

    For every negative set, this function creates:

    - `all_negX`: all training positives plus one matched negative set;
    - `scY_negX`: a leave-one-complex-out collection in which positives from
      complex Y and an equal number of negatives are removed.

    Only training data are stored because evaluation uses the single held-out
    global test set.
    """
    training_sets = {}
    positive_train = list(positive_train)
    positive_train_set = set(positive_train)

    for negative_index, negative_genes in enumerate(
        negative_training_sets, start=1
    ):
        negative_genes = list(negative_genes)

        if len(negative_genes) != len(positive_train):
            raise ValueError(
                f"Negative set {negative_index} contains "
                f"{len(negative_genes)} genes; expected {len(positive_train)}."
            )

        all_genes = positive_train + negative_genes
        permutation = np.random.RandomState(
            random_state + negative_index
        ).permutation(len(all_genes))
        all_genes = [all_genes[index] for index in permutation]

        training_sets[f"all_neg{negative_index}"] = {
            "train": {
                "data": expression_df.loc[all_genes].to_numpy(),
                "classes": [
                    int(gene in positive_train_set) for gene in all_genes
                ],
                "genes": all_genes,
            }
        }

        positive_all = [
            gene for gene in all_genes if gene in positive_train_set
        ]
        negative_all = [
            gene for gene in all_genes if gene not in positive_train_set
        ]

        for complex_index, complex_genes in enumerate(
            positive_subsets, start=1
        ):
            complex_set = set(complex_genes)
            positive_kept = [
                gene for gene in positive_all if gene not in complex_set
            ]
            removed_count = len(positive_all) - len(positive_kept)

            local_rng = random.Random(
                (random_state + 13 * complex_index + 101 * negative_index)
                % (2**31 - 1)
            )
            negatives_removed = set(
                local_rng.sample(negative_all, removed_count)
            )
            negative_kept = [
                gene for gene in negative_all
                if gene not in negatives_removed
            ]

            subset_genes = positive_kept + negative_kept
            subset_permutation = np.random.RandomState(
                random_state + 1000 * negative_index + complex_index
            ).permutation(len(subset_genes))
            subset_genes = [
                subset_genes[index] for index in subset_permutation
            ]

            key = f"sc{complex_index}_neg{negative_index}"
            training_sets[key] = {
                "train": {
                    "data": expression_df.loc[subset_genes].to_numpy(),
                    "classes": [
                        int(gene in positive_train_set)
                        for gene in subset_genes
                    ],
                    "genes": subset_genes,
                }
            }

    return training_sets


## 3. Load input data


In [ ]:
if not EXPRESSION_FILE.exists():
    raise FileNotFoundError(
        f"Expression file not found: {EXPRESSION_FILE.resolve()}"
    )

if not GENE_ANNOTATION_FILE.exists():
    raise FileNotFoundError(
        f"Gene annotation file not found: {GENE_ANNOTATION_FILE.resolve()}"
    )

expression = pd.read_csv(EXPRESSION_FILE, index_col=1)
expression = expression.iloc[:, 1:]
validate_unique_index(expression)

gene_annotation = pd.read_csv(GENE_ANNOTATION_FILE)

required_columns = {"gene_ID_worm", "Complex"}
missing_columns = required_columns.difference(gene_annotation.columns)

if missing_columns:
    raise ValueError(
        "Missing required annotation columns: "
        + ", ".join(sorted(missing_columns))
    )

print(f"Expression matrix: {expression.shape[0]:,} genes × "
      f"{expression.shape[1]:,} features")


## 4. Define positive genes and complex-specific subsets

`dict.fromkeys()` removes duplicates while preserving the original annotation
order. This avoids the non-deterministic ordering produced by converting the
gene list directly to a set.


In [ ]:
training_annotation = gene_annotation.copy()

positive_genes = list(
    dict.fromkeys(training_annotation["gene_ID_worm"].dropna())
)

missing_positive_genes = sorted(
    set(positive_genes).difference(expression.index)
)
if missing_positive_genes:
    raise ValueError(
        f"{len(missing_positive_genes)} positive genes are absent from "
        "the expression matrix."
    )

positive_set = set(positive_genes)
available_negative_genes = [
    gene for gene in expression.index if gene not in positive_set
]

complex_subsets = [
    list(
        dict.fromkeys(
            training_annotation.loc[
                training_annotation["Complex"].eq(complex_id),
                "gene_ID_worm",
            ].dropna()
        )
    )
    for complex_id in ["I", "II", "III", "IV", "V"]
]

print(f"Positive genes: {len(positive_genes)}")
for complex_id, genes_in_complex in zip(
    ["I", "II", "III", "IV", "V"], complex_subsets
):
    print(f"Complex {complex_id}: {len(genes_in_complex)} genes")


## 5. Informed bagging


In [ ]:
(
    X_test,
    y_test,
    test_genes,
    positive_train,
    positive_test,
    negative_test,
) = informed_bagging(
    expression_df=expression,
    positive_genes=positive_genes,
    negative_pool=available_negative_genes,
)

print(f"Training positives: {len(positive_train)}")
print(f"Test positives: {len(positive_test)}")
print(f"Test negatives: {len(negative_test)}")
print(f"Total test genes: {len(test_genes)}")


## 6. Generate negative training sets

Each negative set contains the same number of genes as the positive training
partition. Genes assigned to the held-out test set are excluded to prevent data
leakage.


In [ ]:
negative_training_pool = [
    gene
    for gene in available_negative_genes
    if gene not in negative_test
]

if len(negative_training_pool) < len(positive_train):
    raise ValueError(
        "The available negative pool is too small for balanced training sets."
    )

rng = random.Random(RANDOM_SEED)
negative_training_sets = [
    rng.sample(negative_training_pool, len(positive_train))
    for _ in range(N_NEGATIVE_SETS)
]

training_sets = create_training_dictionaries(
    expression_df=expression,
    positive_train=positive_train,
    positive_subsets=complex_subsets,
    negative_training_sets=negative_training_sets,
)

print(f"Training collections created: {len(training_sets)}")
print(f"Features per gene: "
      f"{training_sets['all_neg1']['train']['data'].shape[1]}")


## 7. Save prepared objects


In [ ]:
test_set = {
    "data": X_test,
    "classes": y_test,
    "genes": test_genes,
}

test_output = OUTPUT_DIR / "01.1_test.pkl"
training_output = OUTPUT_DIR / "01.2_random_training_sets.pkl"

with test_output.open("wb") as file:
    pickle.dump(test_set, file)

with training_output.open("wb") as file:
    pickle.dump(training_sets, file)

print(f"Saved test set to: {test_output}")
print(f"Saved training sets to: {training_output}")


## Output files

- `results/prepared_lists/01.1_test.pkl`: balanced held-out test set.
- `results/prepared_lists/01.2_random_training_sets.pkl`: complete and
  leave-one-complex-out training collections generated with ten negative sets.
